# Digital-twin patient matching -- Eddy spec (5 Jul 2026)

Given a query patient observed up to hour **H**, retrieve the most similar patients from the
training database and use their outcomes and trajectories as a *prognosis by analogy*.

This version follows **Eddy's Step 5 / Step 6 specification** exactly. Similarity is scored from
**three separate, complementary components** (rather than the old feature-cosine + trajectory-cosine
blend):

**Step 5 -- Similarity scoring**
- **State** -- *Hamming distance* on the prototype (IIIC) label sequence up to hour H.
- **Trajectory** -- *cosine similarity* between the twin's transformer hidden states, taken
  **per 6h block up to H and concatenated** into one trajectory vector.
- **Clinical** -- *Euclidean distance* on the normalized clinical variables.
- Each component is min-max normalized per query, then combined with tunable **weights**
  (state / trajectory / clinical). A separate held-out set could be used to *learn* those weights
  (Eddy's "nice to have").
- **Output:** one composite similarity score per training patient -> a ranked list.

**Step 6 -- Retrieval**
- Retrieve the **top-10** most similar training patients.
- Compute the top-10 **outcome distribution** (% good / % poor) and the **mean cohort trajectory**
  (the average roll-forward P(good) curve, plus the mean trajectory embedding).
- Surface the **top-1 match** with **full clinical details** and its **prototype label sequence**
  for direct, bedside case comparison.

The underlying tensors are produced by `model_06_13_2026.ipynb`; this notebook trains nothing.

In [ ]:
import json
import warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D
from sklearn.metrics import roc_auc_score

# pre-cutoff future hours are NaN for every neighbour, so their cohort mean is a (harmless) empty slice
warnings.filterwarnings('ignore', message='Mean of empty slice')

# handoff files produced by model_06_13_2026.ipynb
DIR = Path.home() / 'Desktop/Claude/Capstone_paper/BiLSTM_Keaton/model-06-13-2026'
if not DIR.exists():
    DIR = Path('model-06-13-2026')  # fallback when run from BiLSTM_Keaton/

match = np.load(DIR / 'twin_matching_handoff.npz', allow_pickle=True)
step4 = np.load(DIR / 'twin_step4_handoff.npz', allow_pickle=True)
meta = json.load(open(DIR / 'twin_matching_handoff_meta.json'))

CURRENT_HOURS = list(match['current_hours'])
FUTURE_HOURS = list(match['future_hours'])
CLASS_NAMES = meta['class_names']        # 8 IIIC labels, index == labelseq code
CLIN_COLS = meta['clinical_cols']        # ['age','sex_M','vfib','ROSC(minutes)','time_to_CA(seconds)']
print('current hours:', CURRENT_HOURS)
print('train:', match['train_pids'].shape[0], ' test:', match['test_pids'].shape[0])
print('IIIC labels:', CLASS_NAMES)

In [ ]:
# outcomes, ids, masks, roll-forward outcome trajectories, and the per-block reference curve
y_train = match['y_train']; y_test = match['y_test']
train_pids = match['train_pids']; test_pids = match['test_pids']
mask_train = step4['mask_train']; mask_test = step4['mask_test']

# roll_traj[patient, current_hour_idx, future_hour_idx] = predicted P(good) at the future hour,
# rolled forward from the current hour. NaN where future < current or no EEG observed yet.
roll_train = match['roll_traj_train']; roll_test = match['roll_traj_test']

# the twin's full-data per-block P(good) (real EEG) -- the per-hour reference the forecast checks against
prob_train = step4['outcome_prob_train']; prob_test = step4['outcome_prob_test']

# --- inputs for Eddy's three similarity components ---
# STATE: discrete prototype (IIIC) label per 1h block; -1 == unobserved
labelseq_train = step4['labelseq_train']; labelseq_test = step4['labelseq_test']   # (N, 84) int
# TRAJECTORY: transformer hidden state per 1h block
hidden_train = step4['hidden_train']; hidden_test = step4['hidden_test']           # (N, 84, 256)
# CLINICAL: normalized (z-scored) variables for distance, raw values for display
clin_norm_train = step4['clin_norm_train']; clin_norm_test = step4['clin_norm_test']  # (N, 5)
clin_raw_train = step4['clin_raw_train']; clin_raw_test = step4['clin_raw_test']       # (N, 5)

def hour_to_index(h):
    # column of a current hour in the (N, 14, dim) roll tensors
    return CURRENT_HOURS.index(h)

def hour_to_block(h):
    # hidden states / labelseq are per 1h block; block b corresponds to hour b+1
    return h - 1

N_CLASSES = len(CLASS_NAMES)

## Step 5a -- State similarity (Hamming distance on the prototype label sequence)

Each patient has a discrete IIIC/prototype label per 1h block (`labelseq`, codes 0-7; `-1` where no
EEG yet). For a query observed to hour **H**, the state signal is the **Hamming distance** between
its label sequence `labelseq[:H]` and each training patient's, i.e. the fraction of blocks whose
labels disagree. We compare only positions where **both** patients are observed, so missing data is
not counted as a mismatch; the similarity is `1 - Hamming`. Fully causal (only blocks <= H are used).

In [ ]:
def state_similarity(H):
    # (299 test, 695 train) = 1 - Hamming distance on labelseq[:, :H], over co-observed blocks
    a = labelseq_test[:, :H]           # (Nte, H)
    b = labelseq_train[:, :H]          # (Ntr, H)
    obs_a = (a != -1).astype(np.float32)
    obs_b = (b != -1).astype(np.float32)
    comp = obs_a @ obs_b.T             # (Nte, Ntr) # of co-observed blocks
    # # of co-observed blocks where the labels AGREE = sum over classes of (a==c) @ (b==c)^T
    agree = np.zeros_like(comp)
    for c in range(N_CLASSES):
        agree += (a == c).astype(np.float32) @ (b == c).astype(np.float32).T
    hamming = np.where(comp > 0, (comp - agree) / np.maximum(comp, 1.0), 1.0)
    sim = 1.0 - hamming
    sim[comp == 0] = 0.0               # no overlap -> neutral
    return sim.astype(np.float32)

## Step 5b -- Trajectory similarity (cosine on per-6h concatenated hidden states)

The twin's 256-dim hidden state is sampled at **every 6h block up to H** (hours 6, 12, ..., H) and
those blocks are **concatenated** into a single trajectory vector, which is L2-normalized and
compared by **cosine similarity**. Concatenating the 6h checkpoints (rather than using only the
hour-H block) makes the match sensitive to the *shape* of the whole trajectory, not just its
endpoint. The causal mask guarantees each block only depends on hours <= that block.

In [ ]:
def _l2norm(m):
    # unit-length rows so a dot product equals cosine similarity
    return m / np.clip(np.linalg.norm(m, axis=1, keepdims=True), 1e-8, None)

def trajectory_similarity(H):
    # (299 test, 695 train) cosine of the per-6h-block concatenated hidden-state trajectory up to H
    blocks = [hour_to_block(h) for h in CURRENT_HOURS if h <= H]   # e.g. H=24 -> [5,11,17,23]
    te = hidden_test[:, blocks, :].reshape(hidden_test.shape[0], -1)
    tr = hidden_train[:, blocks, :].reshape(hidden_train.shape[0], -1)
    return _l2norm(te) @ _l2norm(tr).T

## Step 5c -- Clinical similarity (Euclidean distance on normalized variables)

The five clinical variables (age, sex, VFib, ROSC minutes, time-to-CA) are already z-scored
(`clin_norm`). The clinical signal is the **Euclidean distance** in that normalized space; closer
patients are more similar. Clinical values are constant per patient, so this component does not
depend on H.

In [ ]:
def clinical_similarity():
    # (299 test, 695 train) negative Euclidean distance on normalized clinical variables
    d = np.sqrt(((clin_norm_test[:, None, :] - clin_norm_train[None, :, :]) ** 2).sum(2))
    return (-d).astype(np.float32)     # higher (less negative) = closer = more similar

def _minmax_rows(S):
    # scale each query row to [0, 1] so the three heterogeneous components are comparable before weighting
    lo = S.min(1, keepdims=True); hi = S.max(1, keepdims=True)
    return (S - lo) / np.clip(hi - lo, 1e-9, None)

def composite_similarity(H, weights=None):
    # combine the three components (each min-max normalized per query) into one score
    if weights is None:
        weights = DEFAULT_WEIGHTS
    St = _minmax_rows(state_similarity(H))
    Tr = _minmax_rows(trajectory_similarity(H))
    Cl = _minmax_rows(clinical_similarity())
    wsum = weights['state'] + weights['trajectory'] + weights['clinical']
    comp = (weights['state'] * St + weights['trajectory'] * Tr + weights['clinical'] * Cl) / wsum
    return comp, St, Tr, Cl

# default weights (equal). A held-out set could be used to LEARN these (Eddy's 'nice to have').
DEFAULT_WEIGHTS = {'state': 1.0, 'trajectory': 1.0, 'clinical': 1.0}

## Step 5 output + Step 6 -- ranked list and retrieval

`match_patients` produces the **composite ranked list** (Step 5 output) and everything Step 6 asks
for: the **top-k** neighbours, the **outcome distribution** (% good / % poor), the **mean cohort
trajectory** (mean roll-forward P(good) curve + mean trajectory embedding), and the **top-1 case**
with full clinical details and its prototype label sequence. A similarity-weighted `p_good` vote is
also returned for a single-number prognosis.

In [ ]:
def _decode_labelseq(codes):
    # map integer codes to IIIC names; '-' for unobserved (-1)
    return [CLASS_NAMES[c] if c >= 0 else '-' for c in codes]

def match_patients(q_idx, H, k=10, weights=None):
    # Step 5: composite ranked list for test patient q_idx observed up to hour H
    hi = hour_to_index(H)
    comp, St, Tr, Cl = composite_similarity(H, weights)
    row = comp[q_idx]
    order = np.argsort(-row)[:k]                      # Step 6: top-k training neighbours
    w = np.clip(row[order], 1e-6, None)
    outc = y_train[order]

    # outcome distribution over the retrieved cohort
    n_good = int((outc == 1).sum()); n_poor = int((outc == 0).sum())

    # mean cohort trajectory: average roll-forward P(good) curve across neighbours
    cohort_traj = np.nanmean(roll_train[order, hi, :], axis=0)
    # mean trajectory embedding: average of the per-6h concatenated hidden trajectory
    blocks = [hour_to_block(h) for h in CURRENT_HOURS if h <= H]
    mean_embed = hidden_train[np.ix_(order, blocks)].mean(0)   # (n_blocks, 256)

    # top-1 case for direct comparison
    t1 = order[0]
    top1 = dict(
        pid=str(train_pids[t1]),
        outcome='good' if y_train[t1] == 1 else 'poor',
        composite=float(row[t1]),
        clinical={col: float(v) for col, v in zip(CLIN_COLS, clin_raw_train[t1])},
        label_sequence=_decode_labelseq(labelseq_train[t1, :H]),
    )
    return dict(hi=hi, hour=H, neighbors=order, sim=row[order],
                state_sim=St[q_idx, order], traj_sim=Tr[q_idx, order], clin_sim=Cl[q_idx, order],
                outcomes=outc, pids=train_pids[order],
                n_good=n_good, n_poor=n_poor,
                pct_good=100.0 * n_good / k, pct_poor=100.0 * n_poor / k,
                p_good=float(np.average(outc, weights=w)),
                mean_cohort_trajectory=cohort_traj, mean_traj_embedding=mean_embed,
                top1_case=top1)

## Controls

One place to set the component **weights**, the observation cutoff **H**, and the number of
neighbours **K**. `WEIGHTS` are state / trajectory / clinical (they are normalized to sum to 1, so
only their ratio matters).

In [ ]:
# ---- Controls: edit these, then re-run the retrieval / demo cells ----
WEIGHTS = {'state': 1.0, 'trajectory': 2.0, 'clinical': 1.0}  # ratio only; trajectory is the strongest signal
H = 24        # observation cutoff in hours (must be one of CURRENT_HOURS)
K = 10        # number of neighbours to retrieve (Eddy: top-10)
DEFAULT_WEIGHTS = WEIGHTS
print(f'WEIGHTS={WEIGHTS}  H={H}  K={K}')

## Step 6 surfacing -- top-1 case, outcome distribution, mean cohort trajectory

`describe_match` prints exactly the three Step-6 outputs: the **top-1 case** (full clinical detail +
prototype label sequence), the **top-10 outcome distribution**, and a summary of the **mean cohort
trajectory**.

In [ ]:
def _rle(seq):
    # compress a label sequence to 'Label x n -> Label x n' for readability
    out = []
    for lab in seq:
        if out and out[-1][0] == lab:
            out[-1][1] += 1
        else:
            out.append([lab, 1])
    return ' -> '.join(f'{lab}x{n}' if n > 1 else lab for lab, n in out)

def describe_match(q_idx, H=None, k=None, weights=None):
    H = H if H is not None else globals()['H']
    k = k if k is not None else globals()['K']
    res = match_patients(q_idx, H, k, weights)
    truth = 'good' if y_test[q_idx] == 1 else 'poor'
    print(f'QUERY {test_pids[q_idx]}  (true {truth})  observed to {H} h')
    print(f"  query label seq:  {_rle(_decode_labelseq(labelseq_test[q_idx, :H]))}")
    print()
    print(f'  Top-10 outcome distribution:  {res["n_good"]}/{k} good ({res["pct_good"]:.0f}%),  '
          f'{res["n_poor"]}/{k} poor ({res["pct_poor"]:.0f}%)   |   weighted vote P(good) = {res["p_good"]:.3f}')
    cohort = res['mean_cohort_trajectory']
    fh = np.array(FUTURE_HOURS)
    fin = cohort[np.isfinite(cohort)]
    print(f'  Mean cohort trajectory P(good): starts {fin[0]:.2f} -> ends {fin[-1]:.2f} '
          f'(mean over the {k} neighbours\' roll-forward curves)')
    print()
    t1 = res['top1_case']
    print(f'  TOP-1 MATCH {t1["pid"]}  (outcome {t1["outcome"]}, composite {t1["composite"]:.3f})')
    c = t1['clinical']
    sex = 'M' if c['sex_M'] == 1 else 'F'
    print(f'    clinical: age {c["age"]:.0f}, sex {sex}, VFib {int(c["vfib"])}, '
          f'ROSC {c["ROSC(minutes)"]:.0f} min, time-to-CA {c["time_to_CA(seconds)"]:.0f} s')
    print(f'    label seq: {_rle(t1["label_sequence"])}')
    return res

# one poor-outcome and one good-outcome query, observed to H
observed = mask_test[:, :H].sum(1) > 0
q_poor = int(np.where(observed & (y_test == 0))[0][0])
q_good = int(np.where(observed & (y_test == 1))[0][0])
for q in (q_poor, q_good):
    describe_match(q, H, K)
    print('-' * 100)

## Visualization

**Left:** each neighbour's roll-forward P(good) trajectory (green = good, red = poor, opacity by
similarity), the query's own actual trajectory (blue) and twin forecast (dashed black), and the
**mean cohort trajectory** (thick purple). **Right:** the three-component similarity breakdown
(state / trajectory / clinical) for each retrieved neighbour; the title carries the outcome
distribution and vote.

In [ ]:
def plot_match(q_idx, H=None, k=None, weights=None):
    H = H if H is not None else globals()['H']
    k = k if k is not None else globals()['K']
    res = match_patients(q_idx, H, k, weights); hi = res['hi']
    fh = np.array(FUTURE_HOURS, dtype=float)
    hrs = np.arange(1, prob_test.shape[1] + 1)
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5.4), gridspec_kw={'width_ratios': [1.5, 1]})

    # neighbour roll-forward trajectories
    for n, sim in zip(res['neighbors'], res['sim']):
        col = 'C2' if y_train[n] == 1 else 'C3'
        axL.plot(fh, roll_train[n, hi, :], '-', color=col, alpha=float(np.clip(sim, 0.15, 1)), lw=1.4)
    # mean cohort trajectory (Step 6 output)
    axL.plot(fh, res['mean_cohort_trajectory'], '-', color='purple', lw=3, zorder=6, label='mean cohort')
    # query actual (full-data twin) and query forecast (from H)
    axL.plot(hrs, np.where(mask_test[q_idx] > 0, prob_test[q_idx], np.nan), '-', color='C0', lw=1.7, zorder=4)
    axL.plot(fh, roll_test[q_idx, hi, :], '--', color='k', lw=3, zorder=5)
    axL.axvline(H, color='gray', ls=':'); axL.axhline(0.5, color='gray', ls=':', lw=0.8)
    axL.set_xlabel('hours'); axL.set_ylabel('P(good)'); axL.set_ylim(-0.02, 1.02)
    truth = 'good' if y_test[q_idx] == 1 else 'poor'
    axL.set_title(f'{test_pids[q_idx]} (true {truth}) -- {k} neighbours matched at {H} h')
    axL.legend(handles=[Line2D([0], [0], color='C0', lw=1.7, label='query actual (full-data twin)'),
                        Line2D([0], [0], color='k', lw=3, ls='--', label='query forecast (from H)'),
                        Line2D([0], [0], color='purple', lw=3, label='mean cohort trajectory'),
                        Line2D([0], [0], color='C2', lw=2, label='neighbour: good'),
                        Line2D([0], [0], color='C3', lw=2, label='neighbour: poor')],
               fontsize=8, loc='lower left')
    axL.grid(alpha=0.3)

    # three-component similarity breakdown per neighbour
    yloc = np.arange(k)[::-1]
    axR.barh(yloc + 0.25, res['traj_sim'],  height=0.25, color='C0', label='trajectory')
    axR.barh(yloc + 0.00, res['state_sim'], height=0.25, color='C1', label='state')
    axR.barh(yloc - 0.25, res['clin_sim'],  height=0.25, color='C4', label='clinical')
    for yi, n in zip(yloc, res['neighbors']):
        axR.text(0.01, yi + 0.5, f"{train_pids[n]} ({'good' if y_train[n] else 'poor'})", fontsize=7, va='center')
    axR.set_yticks([]); axR.set_xlabel('normalized similarity (0-1)')
    axR.set_title(f"vote P(good)={res['p_good']:.2f} | {res['n_good']}/{k} good, {res['n_poor']}/{k} poor")
    axR.legend(fontsize=8, loc='lower right'); axR.grid(alpha=0.3, axis='x')
    fig.tight_layout(); plt.show()
    return res

for q in (q_poor, q_good):
    plot_match(q, H, K)

## Random patient gallery

A grid of random test patients, each showing neighbour roll-forwards (green/red), the query forecast
(dashed black), and the mean cohort trajectory (purple). Uses the Controls (`WEIGHTS`, `H`, `K`).

In [ ]:
N_RANDOM = 10
NCOLS = 5
SEED = 10

def plot_random_grid(n=None, ncols=None, seed=None, weights=None, h=None, k=None):
    n = N_RANDOM if n is None else n
    ncols = NCOLS if ncols is None else ncols
    seed = SEED if seed is None else seed
    h = H if h is None else h
    k = K if k is None else k

    eligible = np.where(mask_test[:, :h].sum(1) > 0)[0]
    n = min(n, len(eligible))
    picks = np.random.default_rng(seed).choice(eligible, size=n, replace=False)

    ncols = max(1, min(ncols, n)); nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 2.9 * nrows),
                             squeeze=False, sharex=True, sharey=True, layout='constrained')
    axes = axes.ravel()
    fh = np.array(FUTURE_HOURS, dtype=float)
    hrs = np.arange(1, prob_test.shape[1] + 1)

    for ax, q in zip(axes, picks):
        q = int(q)
        res = match_patients(q, h, k, weights); hi = res['hi']
        for nb, sim in zip(res['neighbors'], res['sim']):
            col = 'C2' if y_train[nb] == 1 else 'C3'
            ax.plot(fh, roll_train[nb, hi, :], '-', color=col, alpha=float(np.clip(sim, 0.15, 1)), lw=1.0)
        ax.plot(fh, res['mean_cohort_trajectory'], '-', color='purple', lw=2.2, zorder=6)
        ax.plot(hrs, np.where(mask_test[q] > 0, prob_test[q], np.nan), '-', color='C0', lw=1.6, zorder=4)
        ax.plot(fh, roll_test[q, hi, :], '--', color='k', lw=1.8, zorder=5)
        ax.axvline(h, color='gray', ls=':', lw=0.8); ax.axhline(0.5, color='gray', ls=':', lw=0.8)
        ax.set_ylim(-0.02, 1.02)
        truth = 'good' if y_test[q] == 1 else 'poor'
        ax.set_title(f"{test_pids[q]} ({truth}) | {res['pct_good']:.0f}% good", fontsize=8)
        ax.grid(alpha=0.3)

    for ax in axes[n:]:
        ax.axis('off')
    fig.legend(handles=[Line2D([0], [0], color='C0', lw=1.6, label='query actual'),
                        Line2D([0], [0], color='k', lw=1.8, ls='--', label='query forecast'),
                        Line2D([0], [0], color='purple', lw=2.2, label='mean cohort'),
                        Line2D([0], [0], color='C2', lw=1.5, label='neighbour: good'),
                        Line2D([0], [0], color='C3', lw=1.5, label='neighbour: poor')],
               loc='outside lower center', ncol=5, fontsize=9)
    fig.supxlabel('hours'); fig.supylabel('P(good)')
    fig.suptitle(f'{n} random test patients -- matched at {h} h  (weights={weights or WEIGHTS}, k={k})', fontsize=12)
    plt.show()

plot_random_grid()

## Quantitative validation

Leakage-clean sanity check: predict each test patient's outcome as the unweighted mean outcome of
its `k` nearest **training** neighbours, score AUC vs the truth. Reported for each component alone
and for the composite, at several hours. A **weight sweep** at 24h illustrates Eddy's "nice to have"
-- a held-out set could pick the weight mix that maximizes AUC (here, done manually).

In [ ]:
def neighbor_vote_auc(H, k=25, weights=None):
    comp, St, Tr, Cl = composite_similarity(H, weights)
    obs = mask_test[:, :H].sum(1) > 0; out = {}
    for name, S in [('state', St), ('trajectory', Tr), ('clinical', Cl), ('composite', comp)]:
        order = np.argsort(-S, axis=1)[:, :k]
        out[name] = roc_auc_score(y_test[obs], y_train[order].mean(1)[obs])
    return out

print(f'neighbour-vote outcome AUC (k=25, weights={WEIGHTS}):')
for h in [12, 24, 48, 72]:
    a = neighbor_vote_auc(h, k=25)
    print(f'  h={h:3d}  state={a["state"]:.3f}  trajectory={a["trajectory"]:.3f}  '
          f'clinical={a["clinical"]:.3f}  composite={a["composite"]:.3f}')

print('\nweight sweep @24h  (state:trajectory:clinical) -> composite AUC:')
for w in [(1,0,0),(0,1,0),(0,0,1),(1,1,1),(1,2,1),(1,3,1),(1,4,2)]:
    ww = {'state': w[0], 'trajectory': w[1], 'clinical': w[2]}
    auc = neighbor_vote_auc(24, 25, ww)['composite']
    print(f'  {w[0]}:{w[1]}:{w[2]}  composite AUC={auc:.3f}')

## Notes

- **Three separate components (Eddy Step 5).** State = Hamming on the prototype label sequence;
  trajectory = cosine on per-6h concatenated hidden states; clinical = Euclidean on normalized
  clinical variables. Each is min-max normalized per query, then combined with tunable weights.
- **Retrieval (Eddy Step 6).** Top-10 neighbours, outcome distribution (% good / poor), mean cohort
  trajectory (curve + embedding), and the top-1 case surfaced with full clinical detail and its
  prototype label sequence.
- **Weights are a ratio and can be learned.** Trajectory is the strongest single signal (see the
  validation table), so the default up-weights it; a held-out set could learn the mix formally.
- **Causality.** State uses only blocks <= H; trajectory concatenates only 6h blocks <= H; clinical
  is static. Matching at hour H therefore mimics a real bedside decision at hour H.